## Subagents with LangChain Multi-Agent Systems

### Installing Utilities and Libraries

In [ ]:
%pip install langchain-anthropic==1.5.4 langchain==1.3.14

### Setting up the Environment

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")
anthropic_model_name = os.getenv("ANTHROPIC_MODEL_NAME")

### Instantiating the ChatAnthropic Class

In [ ]:
from langchain_anthropic import ChatAnthropic

model = ChatAnthropic(
    model_name = anthropic_model_name,
    api_key = anthropic_api_key
)

### Create the Web Search Tool

In [ ]:
from anthropic.types import WebSearchTool20260209Param

# Anthropic Web Search tool
search_tool = WebSearchTool20260209Param(
    name="web_search",
    type="web_search_20260209",
    max_uses=3,
)

### Create the Researcher Subagent

In [ ]:
from langchain.agents import create_agent

# Give ONLY the research agent web search
research_model = model.bind_tools([search_tool])

research_agent = create_agent(
    model=research_model,
    tools=[],
    system_prompt="""
You are a senior market research analyst.

Use web search whenever current information,
competitor research, or market trends are required.

Always cite your findings.
"""
)

### Create the Content Write Subagent

In [ ]:
content_writer = create_agent(
    model=model,
    tools=[],
    system_prompt="""
You are an expert Marketing Content Writer.

Create:

- LinkedIn posts
- Product launch announcements
- Marketing copy
- Promotional content

Write in a professional and engaging tone.
"""
)

### Wrap the Subagents as tools

In [ ]:
from langchain.tools import tool

@tool
def research(query: str) -> str:
    """
    Research a topic and return findings.
    """

    print("\n" + "=" * 60)
    print("Executing Research Agent")
    print("=" * 60)
    print(query)
    print()

    result = research_agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": query
                }
            ]
        }
    )

    answer = result["messages"][-1].text

    print("\nResearch Agent Finished.\n")

    return answer


@tool
def write_marketing_copy(prompt: str) -> str:
    """
    Create marketing content.
    """

    print("\n" + "=" * 60)
    print("Executing Content Writer Agent")
    print("=" * 60)
    print(prompt)
    print()

    result = content_writer.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        }
    )

    answer = result["messages"][-1].text

    print("\nContent Writer Finished.\n")

    return answer


### Create the Supervisor Agent

In [ ]:
supervisor = create_agent(
    model=model,

    tools=[
        research,
        write_marketing_copy
    ],

    system_prompt="""
You are the Marketing Supervisor.

Delegate work to the appropriate specialist.

Use:

- research()
    For customer analysis, competitors, personas, positioning.

- write_marketing_copy()
    For LinkedIn posts, launch announcements and promotional content.

Combine the results into one final response.
"""
)

### Invoke the Supervisor Agent

In [ ]:
response = supervisor.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": """
We are launching a new AI-powered fitness smartwatch called FitSense AI.

First identify:

- Target audience
- Customer pain points
- Current fitness wearable trends

Then create a professional LinkedIn launch announcement.
"""
            }
        ]
    }
)

print("\n" + "=" * 60)
print("FINAL RESPONSE")
print("=" * 60)
print(response["messages"][-1].text)